In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import zipfile, os
with zipfile.ZipFile('/content/drive-download-20260612T140644Z-3-001.zip', 'r') as z:
    z.extractall('/content/datasets/')
print(os.listdir('/content/datasets/'))

['SYNTH.npy', 'MP8.npy', 'MP50.npy', 'Rowing.npy', 'CMJ.npy']


In [4]:
!git clone https://github.com/mlgig/FM4MTSC.git
!pip install git+https://github.com/amazon-science/chronos-forecasting.git -q

fatal: destination path 'FM4MTSC' already exists and is not an empty directory.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [5]:
import numpy as np
import torch
import gc
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import accuracy_score, classification_report
from chronos import ChronosPipeline

def get_chronos_embeddings(pipeline, X, batch_size=16):
    all_embeddings = []
    for i in range(0, len(X), batch_size):
        batch = X[i:i+batch_size]
        batch_flat = batch.reshape(batch.shape[0], -1)
        if batch_flat.shape[1] > 512:
            batch_flat = batch_flat[:, :512]
        context = torch.tensor(batch_flat, dtype=torch.float32)
        with torch.no_grad():
            embeddings, _ = pipeline.embed(context)
        all_embeddings.append(embeddings.float().cpu().numpy().reshape(len(batch), -1))
        del embeddings, context, batch_flat
        torch.cuda.empty_cache()
        gc.collect()
        if i % 500 == 0:
            print(f'  {i}/{len(X)} done')
    return np.concatenate(all_embeddings, axis=0)

pipeline = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-tiny",
    device_map="cuda",
    torch_dtype=torch.bfloat16,
)
print('Pipeline ready')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/89 [00:00<?, ?it/s]

Pipeline ready


In [6]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import gc
import torch
torch.cuda.empty_cache()
gc.collect()
print(torch.cuda.get_device_properties(0).total_memory / 1e9, 'GB GPU memory')

15.637086208 GB GPU memory


In [ ]:
name = 'SYNTH'
data = np.load(f'/content/datasets/{name}.npy', allow_pickle=True).item()
X_tr, y_tr = data['train']['X'], np.array([int(x) for x in data['train']['y']])
X_te, y_te = data['test']['X'], np.array([int(x) for x in data['test']['y']])

train_emb = get_chronos_embeddings(pipeline, X_tr)
test_emb = get_chronos_embeddings(pipeline, X_te)
np.save('/content/drive/MyDrive/Chronos_SYNTH_train_emb.npy', train_emb)
np.save('/content/drive/MyDrive/Chronos_SYNTH_test_emb.npy', test_emb)

clf = RidgeClassifier()
clf.fit(train_emb, y_tr)
y_pred = clf.predict(test_emb)
print(f'SYNTH Accuracy: {accuracy_score(y_te, y_pred):.4f}')
print(classification_report(y_te, y_pred))
del train_emb, test_emb, X_tr, X_te, data
gc.collect()
torch.cuda.empty_cache()

  0/7500 done
  2000/7500 done
  4000/7500 done
  6000/7500 done
  0/1000 done


In [ ]:
name = 'CMJ'
data = np.load(f'/content/datasets/{name}.npy', allow_pickle=True).item()
X_tr, y_tr = data['train']['X'], np.array([int(x) for x in data['train']['y']])
X_te, y_te = data['test']['X'], np.array([int(x) for x in data['test']['y']])

train_emb = get_chronos_embeddings(pipeline, X_tr)
test_emb = get_chronos_embeddings(pipeline, X_te)
np.save('/content/drive/MyDrive/Chronos_CMJ_train_emb.npy', train_emb)
np.save('/content/drive/MyDrive/Chronos_CMJ_test_emb.npy', test_emb)

clf = RidgeClassifier()
clf.fit(train_emb, y_tr)
y_pred = clf.predict(test_emb)
print(f'CMJ Accuracy: {accuracy_score(y_te, y_pred):.4f}')
print(classification_report(y_te, y_pred))
del train_emb, test_emb, X_tr, X_te, data
gc.collect()
torch.cuda.empty_cache()

In [ ]:
name = 'MP8'
data = np.load(f'/content/datasets/{name}.npy', allow_pickle=True).item()
X_tr, y_tr = data['train']['X'], np.array([int(x) for x in data['train']['y']])
X_te, y_te = data['test']['X'], np.array([int(x) for x in data['test']['y']])

train_emb = get_chronos_embeddings(pipeline, X_tr)
test_emb = get_chronos_embeddings(pipeline, X_te)
np.save('/content/drive/MyDrive/Chronos_MP8_train_emb.npy', train_emb)
np.save('/content/drive/MyDrive/Chronos_MP8_test_emb.npy', test_emb)

clf = RidgeClassifier()
clf.fit(train_emb, y_tr)
y_pred = clf.predict(test_emb)
print(f'MP8 Accuracy: {accuracy_score(y_te, y_pred):.4f}')
print(classification_report(y_te, y_pred))
del train_emb, test_emb, X_tr, X_te, data
gc.collect()
torch.cuda.empty_cache()

In [ ]:
name = 'MP50'
data = np.load(f'/content/datasets/{name}.npy', allow_pickle=True).item()
X_tr, y_tr = data['train']['X'], np.array([int(x) for x in data['train']['y']])
X_te, y_te = data['test']['X'], np.array([int(x) for x in data['test']['y']])

train_emb = get_chronos_embeddings(pipeline, X_tr)
test_emb = get_chronos_embeddings(pipeline, X_te)
np.save('/content/drive/MyDrive/Chronos_MP50_train_emb.npy', train_emb)
np.save('/content/drive/MyDrive/Chronos_MP50_test_emb.npy', test_emb)

clf = RidgeClassifier()
clf.fit(train_emb, y_tr)
y_pred = clf.predict(test_emb)
print(f'MP50 Accuracy: {accuracy_score(y_te, y_pred):.4f}')
print(classification_report(y_te, y_pred))
del train_emb, test_emb, X_tr, X_te, data
gc.collect()
torch.cuda.empty_cache()